**Natural Language Processing**

**Assignment:** Poetry Generation in Urdu

**Name:** Muhammed Usman

**Student ID:22p-9071**

In [1]:
!pip install -q spacy

In [2]:
import spacy
import random
import re
import zipfile
from collections import Counter, defaultdict
from google.colab import files

In [3]:
uploaded = files.upload()

print("Uploaded files:")
for filename in uploaded:
    print(filename)

Saving faiz.txt to faiz.txt
Saving ghalib.txt to ghalib.txt
Saving iqbal.txt to iqbal.txt
Uploaded files:
faiz.txt
ghalib.txt
iqbal.txt


In [4]:
nlp = spacy.blank("xx")

def tokenize_line(line):
    doc = nlp(line.strip())
    return [
        token.text
        for token in doc
        if not token.is_space and not token.is_punct
    ]

In [5]:
filenames = ["faiz.txt", "ghalib.txt", "iqbal.txt"]

corpus_lines = []
poems = {}

for filename in filenames:
    with open(filename, "r", encoding="utf-8-sig") as file:
        text = file.read()

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    poems[filename] = lines
    corpus_lines.extend(lines)

tokenized_lines = []

for line in corpus_lines:
    tokens = tokenize_line(line)
    if tokens:
        tokenized_lines.append(tokens)

print("Total non-empty lines:", len(corpus_lines))
print("Total tokenized lines:", len(tokenized_lines))
print("Sample tokens:", tokenized_lines[:3])

Total non-empty lines: 2882
Total tokenized lines: 2880
Sample tokens: [['گلوں', 'میں', 'رنگ', 'بھرے', 'باد', 'نوبہار', 'چلے'], ['چلے', 'بھی', 'آؤ', 'کہ', 'گلشن', 'کا', 'کاروبار', 'چلے'], ['قفس', 'اداس', 'ہے', 'یارو', 'صبا', 'سے', 'کچھ', 'تو', 'کہو']]


In [6]:
unigram_counts = Counter()

for tokens in tokenized_lines:
    unigram_counts.update(tokens)

print("Total unique words:", len(unigram_counts))
print("Most common words:")
print(unigram_counts.most_common(20))

Total unique words: 4474
Most common words:
[('ہے', 803), ('میں', 600), ('سے', 496), ('نہ', 394), ('کو', 371), ('کہ', 330), ('کا', 325), ('کے', 304), ('نہیں', 286), ('و', 279), ('کی', 268), ('تو', 267), ('دل', 246), ('کیا', 240), ('تھا', 213), ('ہو', 210), ('ہیں', 200), ('وہ', 194), ('بھی', 191), ('نے', 177)]


In [7]:
forward_bigram = defaultdict(Counter)

for tokens in tokenized_lines:
    for i in range(len(tokens) - 1):
        current_word = tokens[i]
        next_word = tokens[i + 1]

        forward_bigram[current_word][next_word] += 1

print("Number of words with forward transitions:",
      len(forward_bigram))

Number of words with forward transitions: 4198


In [8]:
backward_bigram = defaultdict(Counter)

for tokens in tokenized_lines:
    for i in range(1, len(tokens)):
        current_word = tokens[i]
        previous_word = tokens[i - 1]

        backward_bigram[current_word][previous_word] += 1

print("Number of words with backward transitions:",
      len(backward_bigram))

Number of words with backward transitions: 4092


In [9]:
trigram_counts = defaultdict(Counter)

for tokens in tokenized_lines:
    for i in range(len(tokens) - 2):
        first_word = tokens[i]
        second_word = tokens[i + 1]
        next_word = tokens[i + 2]

        context = (first_word, second_word)
        trigram_counts[context][next_word] += 1

print("Number of trigram contexts:", len(trigram_counts))

Number of trigram contexts: 13397


In [10]:
def choose_word(counter):
    if not counter:
        return None

    words = list(counter.keys())
    weights = list(counter.values())

    return random.choices(words, weights=weights, k=1)[0]

In [11]:
def generate_unigram_line(min_words=7, max_words=10):
    length = random.randint(min_words, max_words)

    words = list(unigram_counts.keys())
    weights = list(unigram_counts.values())

    generated = random.choices(words, weights=weights, k=length)

    return " ".join(generated)


print("UNIGRAM MODEL\n")

for i in range(5):
    print(generate_unigram_line())

UNIGRAM MODEL

ناز اگر تودے بھلا ہے نہیں وائے
نہ نہیں معاملہ ناخن گیا یاد کرشمہ وقت
جس جل لالہ سخن فجر مرے جیسے ھستی و
زخم قطرہ گہ تیرا ہمتِِ اور ناز کہیں
آخر ناز دیکھتے میں مفت ہماری جن اس ہے اگر


In [12]:
def generate_forward_bigram_line(min_words=7, max_words=10):
    length = random.randint(min_words, max_words)

    current_word = choose_word(unigram_counts)
    generated = [current_word]

    for _ in range(length - 1):
        next_word = choose_word(forward_bigram.get(current_word, Counter()))

        if next_word is None:
            current_word = choose_word(unigram_counts)
        else:
            current_word = next_word

        generated.append(current_word)

    return " ".join(generated)


print("FORWARD BIGRAM MODEL\n")

for i in range(5):
    print(generate_forward_bigram_line())

FORWARD BIGRAM MODEL

و نظر نہیں پیمانۂ الا تو سمجھا نہ وہ
کی تابانی سے مجھ کو دیکھتے ہی
را کیا ناصح گر نہیں جلوہ فرما جل
مسیح جس میں رہے قیامت نے چاہا تھا شاید کوئی
طُرفِ کلہ کو سوز پردے میں فقط جلتا ہوں کافر


In [13]:
def generate_trigram_line(min_words=7, max_words=10):
    length = random.randint(min_words, max_words)

    if not trigram_counts:
        return generate_forward_bigram_line(min_words, max_words)

    context = random.choice(list(trigram_counts.keys()))
    generated = [context[0], context[1]]

    while len(generated) < length:
        context = (generated[-2], generated[-1])
        next_word = choose_word(trigram_counts.get(context, Counter()))

        if next_word is None:
            next_word = choose_word(
                forward_bigram.get(generated[-1], Counter())
            )

        if next_word is None:
            next_word = choose_word(unigram_counts)

        generated.append(next_word)

    return " ".join(generated[:length])


print("TRIGRAM MODEL\n")

for i in range(5):
    print(generate_trigram_line())

TRIGRAM MODEL

کبریا کہیں اُس کو حسرتِ پروانہ کا ہے
روحِ فرنگی کو خداوند ہی کہلاؤ خدا اور سہی اور
سرو قد گرمِ خرامِ ناز آ جاوے جہاں
جو ساتھ مرے تا لبِ ساحل آئے نہ بنے بات
دیکھا اسدؔ کو خلوت و محفل ہے


In [14]:
def generate_backward_bigram_line(min_words=7, max_words=10):
    length = random.randint(min_words, max_words)

    current_word = choose_word(unigram_counts)
    generated = [current_word]

    for _ in range(length - 1):
        previous_word = choose_word(
            backward_bigram.get(current_word, Counter())
        )

        if previous_word is None:
            previous_word = choose_word(unigram_counts)

        generated.append(previous_word)
        current_word = previous_word

    generated.reverse()

    return " ".join(generated)


print("BACKWARD BIGRAM MODEL\n")

for i in range(5):
    print(generate_backward_bigram_line())

BACKWARD BIGRAM MODEL

قیامت کا جگر کیا کہیے کہ سانپ فرش
بدگمانی نے پائی دردِ بے جذب مسلمانی سر شک ستم
میں اس کوچے سے کہ شب غم و
دہر کا ذکر کی شکایت ہے غیر
در نہیں ہے ہر حلقہ یاں زنجیر


In [15]:
def generate_bidirectional_line(min_words=7, max_words=10):
    length = random.randint(min_words, max_words)

    middle_word = choose_word(unigram_counts)
    left_side = []
    right_side = []
    current_left = middle_word
    current_right = middle_word

    while len(left_side) + len(right_side) + 1 < length:
        if len(left_side) <= len(right_side):
            previous_word = choose_word(
                backward_bigram.get(current_left, Counter())
            )

            if previous_word is None:
                previous_word = choose_word(unigram_counts)

            left_side.append(previous_word)
            current_left = previous_word

        else:
            next_word = choose_word(
                forward_bigram.get(current_right, Counter())
            )

            if next_word is None:
                next_word = choose_word(unigram_counts)

            right_side.append(next_word)
            current_right = next_word

    generated = list(reversed(left_side)) + [middle_word] + right_side

    return " ".join(generated)


print("BIDIRECTIONAL BIGRAM MODEL\n")

for i in range(5):
    print(generate_bidirectional_line())

BIDIRECTIONAL BIGRAM MODEL

اشیا مرے ہاتھ جو پیماں سے تیز تیز تیز سا
ملَک سب کچھ تو دے طلسم کہن میں
شرمندہٴ معنی نہ ہوا برا کہے کوہ
گے کہ دَیں اس سیم و فراست
رسوا ء قائد اسی کو ہو گئی پھر


In [16]:
def generate_poem(model_function):
    poem = []

    for stanza_number in range(3):
        stanza = []

        for verse_number in range(4):
            verse = model_function(7, 10)
            stanza.append(verse)

        poem.append(stanza)

    return poem


def print_poem(poem, title):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)

    for stanza in poem:
        for verse in stanza:
            print(verse)
        print()


unigram_poem = generate_poem(generate_unigram_line)
forward_poem = generate_poem(generate_forward_bigram_line)
trigram_poem = generate_poem(generate_trigram_line)
backward_poem = generate_poem(generate_backward_bigram_line)
bidirectional_poem = generate_poem(generate_bidirectional_line)

print_poem(unigram_poem, "UNIGRAM POEM")
print_poem(forward_poem, "FORWARD BIGRAM POEM")
print_poem(trigram_poem, "TRIGRAM POEM")
print_poem(backward_poem, "BACKWARD BIGRAM POEM")
print_poem(bidirectional_poem, "BIDIRECTIONAL BIGRAM POEM")


UNIGRAM POEM
نبی نظر کا احکام بے ہوتی جل
ہے نہیں کہ گا پر خوں نکلے
ہوا مر کاوشِ سروش رہ آ نکلا نہ و
کریں پائی احمد اور و جائیں بار

آیا کی کئی لیا تیرا غم خاک خیالِ میں
بے و وجدان سرکش کہ رہے میں احوال سے
عزیز ظن شرارے ہے بھی کو عشق بُو
نہ کا آگے ہے رکھتے شوق بیاباں ساتھ

بات مروت ہے ء معلوم ستارے کہیں بن صرف سے
کیا ہم ہے حرم انجم ساز سکتے ہائے یوں
عہد عشق تو ہاں کوئی جو درخشاں کام کی رہ
میں آئینۂ مجموعہٴ جو غیب و مُرغِ


FORWARD BIGRAM POEM
کب تلک ناخن سے گر نظر ہائے دراز
حباب میں نے فاش کر کہ اس دشت ساماں نکلا
خودی سے طبیعت نے ان کی طرح باغ رضواں سے
راز الفت چھپا کے باب نبرد تھا

تھا مری نوا میں یہ کیا غمخوار
ہاتھوں مر گیا تیرا بھی رکھتا ہوں ایک ہی
میرے نطق نے اک شور اٹھایا غالب ز بس
غزل اپنی جب بھی اگر گیا صدمۂ یک جہاں تمام

تو چھپائے نہ ہوا نہ رکھ اس کی ضعف
و ہم جلو گاہِ خیالِ نازک بدن پر میں
وسعت مرے لیے آنکھ کھل جائے مجھے
سمجھے گا سارا جہان دے تو مجھے کیا کروں گا


TRIGRAM POEM
جا پاؤں غالب کی طرح روشن شمعِ بزمِ
زمانہ با تو نہ دو طعنہ کیا
میری آہ آتشیں سے بالِ عنقا جل
پہنچے گا و

In [17]:
def validate_poem(poem):
    assert len(poem) == 3, "Poem must have 3 stanzas."

    for stanza in poem:
        assert len(stanza) == 4, "Each stanza must have 4 verses."

        for verse in stanza:
            word_count = len(tokenize_line(verse))
            assert 7 <= word_count <= 10, (
                f"Invalid verse length ({word_count}): {verse}"
            )

    print("Validation passed: 3 stanzas, 4 verses each, 7–10 words per verse.")


validate_poem(unigram_poem)
validate_poem(forward_poem)
validate_poem(trigram_poem)
validate_poem(backward_poem)
validate_poem(bidirectional_poem)

Validation passed: 3 stanzas, 4 verses each, 7–10 words per verse.
Validation passed: 3 stanzas, 4 verses each, 7–10 words per verse.
Validation passed: 3 stanzas, 4 verses each, 7–10 words per verse.
Validation passed: 3 stanzas, 4 verses each, 7–10 words per verse.
Validation passed: 3 stanzas, 4 verses each, 7–10 words per verse.


In [18]:
models = {
    "Unigram": generate_unigram_line,
    "Forward Bigram": generate_forward_bigram_line,
    "Trigram": generate_trigram_line,
    "Backward Bigram": generate_backward_bigram_line,
    "Bidirectional Bigram": generate_bidirectional_line
}

for model_name, generator in models.items():
    print("\n" + "-" * 50)
    print(model_name)
    print("-" * 50)

    for _ in range(3):
        print(generator())


--------------------------------------------------
Unigram
--------------------------------------------------
گر ِ سوز مٹ ناز گداز ہے ہو کُلہ
کے ساز پیغام کبھی آگہی زخم ملت روز
ہے میں دنیا میں شور اس ذرۂ اور بو

--------------------------------------------------
Forward Bigram
--------------------------------------------------
پاداشِ عمل کا پيغام ہے جنبش لب سے
الشعراء رسول ہمزہ توف کی خطا کہیں اُس کو
رہا ہے لذت یاب تھا نہ جا سخن کے

--------------------------------------------------
Trigram
--------------------------------------------------
آ کے چمکا گمنام تھا وطن میں منہ
رُک کے نہ مرتا جو زباں کے لیے
پر صحبت مخالف ہے داغ اپنا کلی

--------------------------------------------------
Backward Bigram
--------------------------------------------------
تو پھر بےضابطہ ومن مانی آئینی اس بندۂ حق ہوں
ہے کُشتۂ دشمن ہے جسے منزلوں سے
کاو کاو کاو کاو کاو سخت جانیہائے تنہائی نہ

--------------------------------------------------
Bidirectional Bigram
-------------------------------------------------